# Task 2: Cart Position Control

In Task 1, we stabilized the inverted pendulum at the origin using a parallel PID architecture. Now, the goal is to **move the cart to an arbitrary target position** while keeping the pendulum upright.

The constraint from Task 1 remains: the cart must stay within $|x_c| < 2$ m (the red zones in the animation).

## 1. Dependencies and Setup

In [ ]:
# --- Setup Paths --- #
import sys
sys.path.append("../")
sys.path.append("../model")

# --- General Imports --- #
import numpy as np
import matplotlib.pyplot as plt

# --- Internal Modules --- #
from model.pendulum import InvertedPendulum
from aux.animate import showAnimation

## 2. PID Controller (from Task 1)

We reuse the same `PIDController` class from Task 1 without changes.

In [ ]:
from typing import Callable

class PIDController:
    """Discrete-time PID controller with anti-windup clamp."""

    def __init__(
        self,
        Kp: float,
        Ki: float,
        Kd: float,
        dt: float,
        output_limits: tuple[float, float] = (-np.inf, np.inf),
    ) -> None:
        self.Kp: float = Kp
        self.Ki: float = Ki
        self.Kd: float = Kd
        self.dt: float = dt
        self.output_limits: tuple[float, float] = output_limits
        self.integral: float = 0.0
        self.prev_error: float = 0.0
        self.first_call: bool = True

    def compute(self, error: float) -> float:
        P = self.Kp * error
        self.integral += error * self.dt
        I = self.Ki * self.integral

        if self.first_call:
            D = 0.0
            self.first_call = False
        else:
            D = self.Kd * (error - self.prev_error) / self.dt
        self.prev_error = error

        output = P + I + D
        lo, hi = self.output_limits
        if output > hi:
            self.integral -= error * self.dt
            output = hi
        elif output < lo:
            self.integral -= error * self.dt
            output = lo
        return output

    def reset(self) -> None:
        self.integral = 0.0
        self.prev_error = 0.0
        self.first_call = True

## 3. The Position Tracking Problem

In Task 1 the cart PID held the cart at $x_c = 0$. To move the cart, we simply change the target setpoint $x_c^{\text{ref}}$ during the simulation.

**How it works mechanically:** when $x_c^{\text{ref}}$ changes to a new value (say $+1$ m), the cart PID generates a force that intentionally *tilts* the pendulum in the desired direction. The much stronger angle PID responds by accelerating the cart toward the target. In essence, the cart PID "steers" through the pendulum dynamics.

**Potential issue — derivative kick:** a step change in $x_c^{\text{ref}}$ causes a one-step spike in the D-term of the cart PID. The actuator clips this to 30 N, so it lasts only one timestep (10 ms). The angle PID absorbs the transient without problems.

**Tuning for position tracking:** the Task 1 gains ($K_p^x = -5$, $K_i^x = -0.05$, $K_d^x = -5$) were tuned for stabilization — the cart returned to the origin slowly because that was fine for stabilization. For position tracking, we need faster convergence. Increasing the cart PID gains speeds up the response but also causes larger pendulum tilts during the transition. There is a hard limit: if the cart PID becomes too aggressive, its tilting force overwhelms the angle PID and the system destabilizes.

**Gains (retuned for Task 2):**

| PID | $K_p$ | $K_i$ | $K_d$ |
|-----|--------|--------|--------|
| Angle | 150 | 0.5 | 15 |
| Cart | −10 | −0.30 | −8 |

Compared to Task 1: $K_p^x$ doubled, $K_i^x$ increased 6×, $K_d^x$ increased 1.6×. This reduces settling time from ~16 s to ~11 s, at the cost of larger transient pendulum angles (~8° instead of ~4°).

**Reachable range:** step targets up to $|x_c^{\text{ref}}| \approx 1.4$ m are safe. Beyond that, the overshoot during the transient can exceed the ±2 m constraint.

### Trajectory Helper

Instead of hard-coding `if`/`elif` chains for the reference signal, we define a reusable function. It takes a list of `(time, position)` waypoints and returns a callable:

In [ ]:
def make_trajectory(waypoints: list[tuple[float, float]]) -> Callable[[float], float]:
    """Piecewise-constant reference from [(t, xc_ref), ...] waypoints."""
    def ref(t: float) -> float:
        val: float = waypoints[0][1]
        for t_switch, v in waypoints:
            if t >= t_switch:
                val = v
        return val
    return ref

## 4. Step Change Demo

The simplest test: start at the origin, then at $t = 5$ s change the target to $x_c^{\text{ref}} = 1.0$ m.

In [ ]:
dt = 0.01
timeSpan = 30
nSteps = int(timeSpan / dt)

x0 = [0, 0, 0.0, 0]  # start perfectly upright at origin
pendulum = InvertedPendulum(x0, dt=dt)

pid_theta = PIDController(Kp=150, Ki=0.5, Kd=15, dt=dt)
pid_cart  = PIDController(Kp=-10, Ki=-0.30, Kd=-8, dt=dt)

theta_target = 0.0

# --- Define reference trajectory --- #
trajectory = make_trajectory([(0, 0.0), (5, 1.0)])

# --- Storage --- #
y_cl  = np.zeros((2, nSteps))
u_log = np.zeros((2, nSteps))
t_cl  = np.zeros(nSteps)
ref_log = np.zeros(nSteps)

y_cl[:, 0] = [x0[0], x0[2]]

# --- Simulation loop --- #
for i in range(1, nSteps):
    t_now = i * dt
    xc_target = trajectory(t_now)
    ref_log[i] = xc_target

    xc_current    = y_cl[0, i-1]
    theta_current = y_cl[1, i-1]

    F_theta = pid_theta.compute(theta_target - theta_current)
    F_cart  = pid_cart.compute(xc_target - xc_current)
    F = np.clip(F_theta + F_cart, -30, 30)

    y_tmp, uRealized = pendulum.step(F)
    y_cl[:, i]  = y_tmp
    u_log[0, i] = F
    u_log[1, i] = uRealized
    t_cl[i]     = t_now

print(f"Final cart position: xc = {y_cl[0, -1]:.4f} m (target: 1.0 m)")
print(f"Final angle: θ = {y_cl[1, -1]:.6f} rad")
print(f"Max |xc| = {np.max(np.abs(y_cl[0])):.4f} m (limit: 2.0 m)")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# --- Cart position --- #
axes[0].plot(t_cl, y_cl[0], color='tab:blue', label='$x_c$')
axes[0].plot(t_cl, ref_log, 'k--', alpha=0.5, label='$x_c^{ref}$')
axes[0].axhline(2, color='r', linestyle='--', alpha=0.3, label='constraint')
axes[0].axhline(-2, color='r', linestyle='--', alpha=0.3)
axes[0].set_ylabel("Cart position [m]")
axes[0].legend(loc='upper left')
axes[0].grid(alpha=0.3)
axes[0].set_title("Step Change: $x_c^{ref} = 0 \\to 1.0$ m at $t = 5$ s")

# --- Pendulum angle --- #
axes[1].plot(t_cl, np.degrees(y_cl[1]), color='tab:orange')
axes[1].set_ylabel("Pendulum angle [deg]")
axes[1].grid(alpha=0.3)

# --- Force --- #
axes[2].plot(t_cl, u_log[0], color='tab:green', alpha=0.7, label='commanded')
axes[2].plot(t_cl, u_log[1], color='tab:red', alpha=0.5, label='realized')
axes[2].axhline(30, color='r', linestyle=':', alpha=0.3)
axes[2].axhline(-30, color='r', linestyle=':', alpha=0.3)
axes[2].set_ylabel("Force [N]")
axes[2].set_xlabel("Time [s]")
axes[2].legend(loc='upper right')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
showAnimation(t_cl, y_cl[0], y_cl[1])

## 5. Multi-Step Reference Tracking

A more realistic scenario: the cart follows a sequence of target positions.

$$x_c^{\text{ref}}(t) = \begin{cases} 0 & t < 5 \\ 1.0 & 5 \leq t < 12 \\ -1.0 & 12 \leq t < 20 \\ 0 & t \geq 20 \end{cases}$$

In [ ]:
dt = 0.01
timeSpan = 35
nSteps = int(timeSpan / dt)

x0 = [0, 0, 0.0, 0]
pendulum = InvertedPendulum(x0, dt=dt)

pid_theta = PIDController(Kp=150, Ki=0.5, Kd=15, dt=dt)
pid_cart  = PIDController(Kp=-10, Ki=-0.30, Kd=-8, dt=dt)

theta_target = 0.0

# --- Multi-step trajectory: just list the waypoints --- #
trajectory = make_trajectory([(0, 0.0), (5, 1.0), (12, -1.0), (20, 0.0)])

y_cl  = np.zeros((2, nSteps))
u_log = np.zeros((2, nSteps))
t_cl  = np.zeros(nSteps)
ref_log = np.zeros(nSteps)

y_cl[:, 0] = [x0[0], x0[2]]

for i in range(1, nSteps):
    t_now = i * dt
    xc_target = trajectory(t_now)
    ref_log[i] = xc_target

    xc_current    = y_cl[0, i-1]
    theta_current = y_cl[1, i-1]

    F_theta = pid_theta.compute(theta_target - theta_current)
    F_cart  = pid_cart.compute(xc_target - xc_current)
    F = np.clip(F_theta + F_cart, -30, 30)

    y_tmp, uRealized = pendulum.step(F)
    y_cl[:, i]  = y_tmp
    u_log[0, i] = F
    u_log[1, i] = uRealized
    t_cl[i]     = t_now

print(f"Final cart position: xc = {y_cl[0, -1]:.4f} m (target: 0.0 m)")
print(f"Max |xc| = {np.max(np.abs(y_cl[0])):.4f} m")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(t_cl, y_cl[0], color='tab:blue', label='$x_c$')
axes[0].plot(t_cl, ref_log, 'k--', alpha=0.5, label='$x_c^{ref}$')
axes[0].axhline(2, color='r', linestyle='--', alpha=0.3)
axes[0].axhline(-2, color='r', linestyle='--', alpha=0.3)
axes[0].set_ylabel("Cart position [m]")
axes[0].legend(loc='upper left')
axes[0].grid(alpha=0.3)
axes[0].set_title("Multi-Step Reference: $0 \\to 1 \\to -1 \\to 0$ m")

axes[1].plot(t_cl, np.degrees(y_cl[1]), color='tab:orange')
axes[1].set_ylabel("Pendulum angle [deg]")
axes[1].grid(alpha=0.3)

axes[2].plot(t_cl, u_log[0], color='tab:green', alpha=0.7, label='commanded')
axes[2].plot(t_cl, u_log[1], color='tab:red', alpha=0.5, label='realized')
axes[2].axhline(30, color='r', linestyle=':', alpha=0.3)
axes[2].axhline(-30, color='r', linestyle=':', alpha=0.3)
axes[2].set_ylabel("Force [N]")
axes[2].set_xlabel("Time [s]")
axes[2].legend(loc='upper right')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
showAnimation(t_cl, y_cl[0], y_cl[1])

## 6. Performance Summary

Key metrics for the step change demo ($0 \to 1.0$ m at $t = 5$ s):

In [ ]:
# Re-run step change to compute metrics
dt = 0.01
timeSpan = 30
nSteps = int(timeSpan / dt)

x0 = [0, 0, 0.0, 0]
pendulum = InvertedPendulum(x0, dt=dt)
pid_theta = PIDController(Kp=150, Ki=0.5, Kd=15, dt=dt)
pid_cart  = PIDController(Kp=-10, Ki=-0.30, Kd=-8, dt=dt)

trajectory = make_trajectory([(0, 0.0), (5, 1.0)])

y_cl  = np.zeros((2, nSteps))
u_log = np.zeros(nSteps)
t_cl  = np.zeros(nSteps)
y_cl[:, 0] = [x0[0], x0[2]]

for i in range(1, nSteps):
    t_now = i * dt
    xc_target = trajectory(t_now)

    F_theta = pid_theta.compute(0.0 - y_cl[1, i-1])
    F_cart  = pid_cart.compute(xc_target - y_cl[0, i-1])
    F = np.clip(F_theta + F_cart, -30, 30)
    y_tmp, _ = pendulum.step(F)
    y_cl[:, i] = y_tmp
    u_log[i] = F
    t_cl[i] = t_now

# Metrics after the step at t=5s
step_idx = int(5.0 / dt)
xc_after = y_cl[0, step_idx:]
t_after = t_cl[step_idx:]
theta_after = y_cl[1, step_idx:]

# Rise time: 10% to 90% of step
target = 1.0
start_val = y_cl[0, step_idx]
rise_10 = start_val + 0.1 * (target - start_val)
rise_90 = start_val + 0.9 * (target - start_val)
t_10 = t_after[np.argmax(xc_after >= rise_10)] - 5.0
t_90 = t_after[np.argmax(xc_after >= rise_90)] - 5.0
rise_time = t_90 - t_10

# Settling time (within 2% of target)
threshold = 0.02 * abs(target)
settled = np.abs(xc_after - target) < threshold
for j in range(len(settled) - 1, -1, -1):
    if not settled[j]:
        settling_time = t_after[j] - 5.0
        break
else:
    settling_time = 0.0

# Overshoot
overshoot_abs = np.max(xc_after) - target
overshoot_pct = max(0, overshoot_abs / abs(target) * 100)

# Max angle during transition
max_angle_deg = np.max(np.abs(theta_after)) * 180 / np.pi

# Steady-state error
ss_error = abs(y_cl[0, -1] - target)

# Max cart excursion
max_xc = np.max(np.abs(y_cl[0]))

# Build table
metrics = [
    ["Rise time (10%→90%)", f"{rise_time:.2f} s"],
    ["Settling time (2%)", f"{settling_time:.2f} s"],
    ["Overshoot", f"{overshoot_pct:.1f}%"],
    ["Steady-state error", f"{ss_error:.4f} m"],
    ["Max |θ| during move", f"{max_angle_deg:.2f}°"],
    ["Max |xc|", f"{max_xc:.3f} m"],
    ["Constraint violated", "No" if max_xc < 2.0 else "YES"],
]

fig, ax = plt.subplots(figsize=(6, 3))
ax.axis('off')
table = ax.table(
    cellText=metrics,
    colLabels=["Metric", "Value"],
    loc='center',
    cellLoc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.6)

for j in range(2):
    cell = table[0, j]
    cell.set_facecolor('#4a6aa5')
    cell.set_text_props(color='white', fontweight='bold')

for i in range(1, len(metrics) + 1):
    for j in range(2):
        cell = table[i, j]
        cell.set_facecolor('#f0f4fa' if i % 2 == 0 else 'white')

plt.title("Step Response Metrics ($0 \\to 1.0$ m)", fontsize=13, pad=20)
plt.tight_layout()
plt.show()

## 7. Conclusion

The parallel PID architecture from Task 1 handles position tracking with only **minor retuning** of the cart PID gains. The core change is making $x_c^{\text{ref}}$ time-varying instead of constant.

**Key observations:**

- **Step changes work reliably.** The cart reaches the target position with near-zero steady-state error thanks to the integral term.
- **The pendulum tilts briefly during moves** — this is physically necessary. The cart can only accelerate by leaning the pendulum, so transient angle excursions (~8°) are expected.
- **Cart PID gains were increased** ($K_p^x: -5 \to -10$, $K_i^x: -0.05 \to -0.30$, $K_d^x: -5 \to -8$) to reduce settling time from ~16 s to ~11 s. The tradeoff is larger transient pendulum angles.
- **Fundamental speed limit:** the cart PID works *through* the pendulum dynamics (tilting to move), so it cannot be made arbitrarily fast without destabilizing the angle control. This is an inherent limitation of the parallel PID approach.
- **Constraint satisfaction is maintained.** For targets up to $|x_c^{\text{ref}}| \leq 1.4$ m, the cart stays within the ±2 m limits.